# Module 02: Symmetric Cryptography & Hash Functions — Lab

This lab provides hands-on exploration of AES internals, SHA-2/SHA-3 hash functions, and HMAC computation.

**Objectives:**
1. Compute AES S-box values and understand its non-linearity
2. Trace a single AES round operation
3. Compute SHA-256 and SHA-3 hashes
4. Implement and test HMAC

In [ ]:
import numpy as np
import hashlib
import hmac
import struct

# AES S-box (complete 256-byte lookup table)
AES_SBOX = [
    0x63, 0x7C, 0x77, 0x7B, 0xF2, 0x6B, 0x6F, 0xC5, 0x30, 0x01, 0x67, 0x2B, 0xFE, 0xD7, 0xAB, 0x76,
    0xCA, 0x82, 0xC9, 0x7D, 0xFA, 0x59, 0x47, 0xF0, 0xAD, 0xD4, 0xA2, 0xAF, 0x9C, 0xA4, 0x72, 0xC0,
    0xB7, 0xFD, 0x93, 0x26, 0x36, 0x3F, 0xF7, 0xCC, 0x34, 0xA5, 0xE5, 0xF1, 0x71, 0xD8, 0x31, 0x15,
    0x04, 0xC7, 0x23, 0xC3, 0x18, 0x96, 0x05, 0x9A, 0x07, 0x12, 0x80, 0xE2, 0xEB, 0x27, 0xB2, 0x75,
    0x09, 0x83, 0x2C, 0x1A, 0x1B, 0x6E, 0x5A, 0xA0, 0x52, 0x3B, 0xD6, 0xB3, 0x29, 0xE3, 0x2F, 0x84,
    0x53, 0xD1, 0x00, 0xED, 0x20, 0xFC, 0xB1, 0x5B, 0x6A, 0xCB, 0xBE, 0x39, 0x4A, 0x4C, 0x58, 0xCF,
    0xD0, 0xEF, 0xAA, 0xFB, 0x43, 0x4D, 0x33, 0x85, 0x45, 0xF9, 0x02, 0x7F, 0x50, 0x3C, 0x9F, 0xA8,
    0x51, 0xA3, 0x40, 0x8F, 0x92, 0x9D, 0x38, 0xF5, 0xBC, 0xB6, 0xDA, 0x21, 0x10, 0xFF, 0xF3, 0xD2,
    0xCD, 0x0C, 0x13, 0xEC, 0x5F, 0x97, 0x44, 0x17, 0xC4, 0xA7, 0x7E, 0x3D, 0x64, 0x5D, 0x19, 0x73,
    0x60, 0x81, 0x4F, 0xDC, 0x22, 0x2A, 0x90, 0x88, 0x46, 0xEE, 0xB8, 0x14, 0xDE, 0x5E, 0x0B, 0xDB,
    0xE0, 0x32, 0x3A, 0x0A, 0x49, 0x06, 0x24, 0x5C, 0xC2, 0xD3, 0xAC, 0x62, 0x91, 0x95, 0xE4, 0x79,
    0xE7, 0xC8, 0x37, 0x6D, 0x8D, 0xD5, 0x4E, 0xA9, 0x6C, 0x56, 0xF4, 0xEA, 0x65, 0x7A, 0xAE, 0x08,
    0xBA, 0x78, 0x25, 0x2E, 0x1C, 0xA6, 0xB4, 0xC6, 0xE8, 0xDD, 0x74, 0x1F, 0x4B, 0xBD, 0x8B, 0x8A,
    0x70, 0x3E, 0xB5, 0x66, 0x48, 0x03, 0xF6, 0x0E, 0x61, 0x35, 0x57, 0xB9, 0x86, 0xC1, 0x1D, 0x9E,
    0xE1, 0xF8, 0x98, 0x11, 0x69, 0xD9, 0x8E, 0x94, 0x9B, 0x1E, 0x87, 0xE9, 0xCE, 0x55, 0x28, 0xDF,
    0x8C, 0xA1, 0x89, 0x0D, 0xBF, 0xE6, 0x42, 0x68, 0x41, 0x99, 0x2D, 0x0F, 0xB0, 0x54, 0xBB, 0x16,
]

# AES S-box analysis
print("=" * 60)
print("AES S-BOX ANALYSIS")
print("=" * 60)

hw = lambda b: bin(b).count('1')

# Compute Hamming weight distribution of S-box outputs
hw_dist = [0] * 9  # HW can be 0-8
for val in AES_SBOX:
    hw_dist[hw(val)] += 1

print("Hamming Weight Distribution of S-box outputs:")
for i, count in enumerate(hw_dist):
    bar = '#' * count
    print(f"  HW={i}: {count:3d} values  {bar}")

print(f"\nS-box output range: [{min(AES_SBOX)}, {max(AES_SBOX)}]")
print(f"Mean S-box output: {np.mean(AES_SBOX):.2f}")
print(f"Std dev of S-box output: {np.std(AES_SBOX):.2f}")

In [ ]:
# Trace a single AES round operation
def gf_mul(a, b):
    """Multiply in GF(2^8) with AES irreducible polynomial x^8 + x^4 + x^3 + x + 1"""
    p = 0
    for _ in range(8):
        if b & 1:
            p ^= a
        high_bit = a & 0x80
        a = (a << 1) & 0xFF
        if high_bit:
            a ^= 0x1B
        b >>= 1
    return p

def mix_columns(state):
    """Apply MixColumns to a 4x4 state matrix"""
    matrix = [
        [0x02, 0x03, 0x01, 0x01],
        [0x01, 0x02, 0x03, 0x01],
        [0x01, 0x01, 0x02, 0x03],
        [0x03, 0x01, 0x01, 0x02],
    ]
    result = [[0]*4 for _ in range(4)]
    for col in range(4):
        for row in range(4):
            val = 0
            for k in range(4):
                val ^= gf_mul(matrix[row][k], state[k][col])
            result[row][col] = val
    return result

# Example state (after SubBytes + ShiftRows)
test_state = [
    [0x19, 0xA0, 0x9A, 0xB1],
    [0xA1, 0xC6, 0x6F, 0x38],
    [0xBD, 0x62, 0xF1, 0x6A],
    [0xE3, 0x82, 0x11, 0x6D],
]

print("=" * 60)
print("AES MixColumns OPERATION")
print("=" * 60)

print("\nInput State:")
for row in test_state:
    print(" ".join(f"{b:02X}" for b in row))

mixed = mix_columns(test_state)

print("\nOutput State (after MixColumns):")
for row in mixed:
    print(" ".join(f"{b:02X}" for b in row))

# Verify: GF multiplication examples
print("\nGF(2^8) Multiplication Examples:")
print(f"  0x02 * 0x53 = 0x{gf_mul(0x02, 0x53):02X}")
print(f"  0x03 * 0x53 = 0x{gf_mul(0x03, 0x53):02X}")
print(f"  0x02 * 0xCA = 0x{gf_mul(0x02, 0xCA):02X}")

In [ ]:
# SHA-256 and SHA-3 verification
print("=" * 60)
print("HASH FUNCTION VERIFICATION")
print("=" * 60)

# Test vectors from NIST
test_messages = {
    "empty": b"",
    "abc": b"abc",
    "896TB": b"abcdbcdecdefdefgefghfghighijhijkijkljklmklmnlmnomnopnopq",
}

# NIST SHA-256 test vectors
sha256_expected = {
    "empty": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "abc": "ba7816bf8f01cfea414140de5dae2223b00361a396177a9cb410ff61f20015ad",
}

for name, msg in test_messages.items():
    digest = hashlib.sha256(msg).hexdigest()
    expected = sha256_expected.get(name, "N/A")
    status = "PASS" if digest == expected else "CHECK"
    print(f"\n[{status}] SHA-256({name}):")
    print(f"  Computed: {digest}")
    if expected != "N/A":
        print(f"  Expected: {expected}")

# SHA-3 verification
print("\n" + "=" * 60)
print("SHA-3 (Keccak) Verification")
print("=" * 60)

for name, msg in test_messages.items():
    digest = hashlib.sha3_256(msg).hexdigest()
    print(f"\nSHA3-256({name}):")
    print(f"  {digest}")

# SHAKE (extendable output)
shake_output = hashlib.shake_256(b"test").hexdigest(32)
print(f"\nSHAKE-256(b\"test\", 32 bytes):")
print(f"  {shake_output}")

In [ ]:
# HMAC implementation and verification
print("=" * 60)
print("HMAC IMPLEMENTATION")
print("=" * 60)

def hmac_sha256(key, message):
    """Manual HMAC-SHA256 implementation"""
    block_size = 64  # SHA-256 block size
    
    # Process key
    if len(key) > block_size:
        key = hashlib.sha256(key).digest()
    key = key.ljust(block_size, b'\x00')
    
    # ipad and opad
    ipad_key = bytes(k ^ 0x36 for k in key)
    opad_key = bytes(k ^ 0x5C for k in key)
    
    # Inner hash
    inner = hashlib.sha256(ipad_key + message).digest()
    
    # Outer hash
    return hashlib.sha256(opad_key + inner).digest()

# Test with known key and message
test_key = b"secret-key-123"
test_msg = b"Hello, Side-Channel Analysis!"

manual_hmac = hmac_sha256(test_key, test_msg)
library_hmac = hmac.new(test_key, test_msg, hashlib.sha256).digest()

print(f"Key: {test_key.decode()}")
print(f"Message: {test_msg.decode()}")
print(f"\nManual HMAC-SHA256:  {manual_hmac.hex()}")
print(f"Library HMAC-SHA256: {library_hmac.hex()}")
print(f"Match: {'PASS' if manual_hmac == library_hmac else 'FAIL'}")

# Verify NIST HMAC test vector (RFC 4231 Test Case 2)
rfc4231_key = bytes([0x0b] * 20)
rfc4231_msg = b"Hi There"
rfc4231_expected = "b0344c61d8db38535ca8afceafd0f0e5"

result = hmac.new(rfc4231_key, rfc4231_msg, hashlib.md5).hexdigest()
print(f"\nRFC 4231 Test Case 2 (HMAC-MD5):")
print(f"  Computed: {result}")
print(f"  Expected: {rfc4231_expected}")
print(f"  Status: {'PASS' if result == rfc4231_expected else 'FAIL'}")